# Few-Shot Prompting Baselines for Idiom-Aware EN-DE Translation

This notebook adds a **prompted baseline track** to our idiom-aware MT project.

It is designed to fit the workflow of the other notebooks:
- mount Drive and define shared paths
- load the idiom dataset used elsewhere in the project
- run prompted generation on a fixed idiom evaluation subset
- score outputs with BLEU / chrF
- export predictions and metrics for later analysis

We start with **mT5** and keep the notebook parameterized so the same workflow can be rerun later with **FLAN-T5** by changing a single checkpoint variable.


## Clone Repo from Git / Commit Changes
Run the git clone every time a new session is started.


In [ ]:

!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls


## Mount Drive, Create Paths


In [1]:

from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")
QUAL_PREDS_DIR = os.path.join(DRIVE_ROOT, "qual_preds")
PROMPT_RESULTS_DIR = os.path.join(RESULTS_DIR, "prompting")

for p in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR, QUAL_PREDS_DIR, PROMPT_RESULTS_DIR]:
    os.makedirs(p, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Prompt results:", PROMPT_RESULTS_DIR)


Mounted at /content/drive
Drive root: /content/drive/MyDrive/ds266_idiom_mt
Prompt results: /content/drive/MyDrive/ds266_idiom_mt/results/prompting


## Install Dependencies & Imports

Note: transformers will load in mT5 along with the typical generation and tokenizers. Also, protobuf>6 is needed due to some wonky issues with hugging face tokenizer crashes and model loading errors. Staying consistent with the other notebooks, we can say that all experiments were run using a consistent Hugging Face environment with pinned protobuf version (<6) to ensure reproducibility.

In [2]:

!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00


In [3]:

import os, random, json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import torch
import sacrebleu
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cpu


## Experiment Configuration

Start with **mT5**. Later, to test FLAN-T5, change only `MODEL_NAME` and rerun.


In [4]:

# --- model checkpoint ---
MODEL_NAME = "google/mt5-small"
RUN_PREFIX = "mt5_small_prompt"

# --- prompting setup ---
SHOT_COUNTS = [0, 3, 5]
EVAL_N = 25
MAX_SOURCE_LEN = 512
MAX_NEW_TOKENS = 128
BATCH_SIZE = 4
DO_SAMPLE = False
NUM_BEAMS = 4

# --- decoding kwargs ---
GEN_KWARGS = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": NUM_BEAMS,
    "do_sample": DO_SAMPLE,
}

print(MODEL_NAME)
print("shot counts:", SHOT_COUNTS)


google/mt5-small
shot counts: [0, 3, 5]


## Load Data

We reuse the idiom dataset from the existing experiment notebooks.


In [5]:

def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)
    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src", "tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

idiom_ds = load_idioms(seed=SEED)
{k: len(v) for k, v in idiom_ds.items()}


README.md: 0.00B [00:00, ?B/s]

en-de.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'train': 800, 'dev': 100, 'test': 100}

In [6]:

# Fixed evaluation subset for reproducibility
idiom_test_df = idiom_ds["test"].to_pandas().copy()
idiom_eval_df = idiom_test_df.sample(n=min(EVAL_N, len(idiom_test_df)), random_state=SEED).reset_index(drop=True)
idiom_eval_df.head()


,src,tgt
0,I'm as fit as a fiddle-with energy to spare.,Ich bin fit wie ein Turnschuh - und habe noch ...
1,Those cheap little metal cars are a dime a dozen.,Diese billigen kleinen Metallautos gibt es wie...
2,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...
3,How long does it take for the wheel of fashion...,"Wie lange dauert es, bis sich der Kreis in der..."
4,"For a sustained economic recovery, the US cons...",Für einen nachhaltigen Wirtschaftsaufschwung m...


## Prompt Helpers

We build prompts from train-set exemplars. For few-shot runs, we sample examples from the idiom train split.

Important note:
- This is **few-shot prompting / in-context learning**, not prompt tuning in the parameter-learning sense.
- Because mT5 is not instruction-tuned, performance may be unstable. That is part of what makes the comparison interesting.


In [7]:

def build_demo_block(example):
    return (
        "Translate the English sentence into natural German.\n\n"
        f"English: {example['src']}\n\n"
        f"German: {example['tgt']}"
    )


def sample_demos(train_df, n, exclude_src=None, seed=42):
    pool = train_df if exclude_src is None else train_df[train_df["src"] != exclude_src]
    if n == 0:
        return []
    return pool.sample(n=n, random_state=seed).to_dict("records")


def build_prompt(src_text, demos=None):
    demos = demos or []
    header = (
        "You are translating English idiomatic sentences into fluent, "
        "meaning-preserving German. Do not translate idioms word-for-word "
        "when a natural German rendering is more appropriate."
    )

    if len(demos) == 0:
        return (
            f"{header}\n\n"
            "Translate the English sentence into natural German.\n\n"
            f"English: {src_text}\n\n"
            "German:"
        )

    demo_text = "\n\n".join(build_demo_block(ex) for ex in demos)

    return (
        f"{header}\n\n"
        f"Examples:\n\n{demo_text}\n\n"
        "Now translate the next sentence.\n\n"
        f"English: {src_text}\n\n"
        "German:"
    )

In [8]:

idiom_train_df = idiom_ds["train"].to_pandas().copy()

# quick sanity check
example_row = idiom_eval_df.iloc[0]
example_demos = sample_demos(idiom_train_df, 3, exclude_src=example_row["src"], seed=SEED)
print(build_prompt(example_row["src"], example_demos)[:1600])


You are translating English idiomatic sentences into fluent, meaning-preserving German. Do not translate idioms word-for-word when a natural German rendering is more appropriate.

Examples:

Translate the English sentence into natural German.

English: Don't discount any job, see it as an opportunity to get a foot in the door.

German: Schlagen Sie keine Stelle aus, sondern sehen Sie sie als Chance, einen Fuß in die Tür zu bekommen.

Translate the English sentence into natural German.

English: Because I'm in the same boat.

German: Ich sitze nämlich im selben Boot.

Translate the English sentence into natural German.

English: Are you not biting off more than you can chew with this project?

German: Übernimmst du dich nicht mit diesem Projekt?

Now translate the next sentence.

English: I'm as fit as a fiddle-with energy to spare.

German:


## Load Prompted Model


In [9]:

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# NOTE: mT5 may emit tied-weight warnings on load; safe to ignore for inference.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

loaded: google/mt5-small


## Generation Helpers


In [10]:

@torch.no_grad()
def generate_texts(prompts, batch_size=BATCH_SIZE, gen_kwargs=None):
    gen_kwargs = gen_kwargs or {}
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        enc = tok(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(device)
        out = model.generate(**enc, **gen_kwargs)
        batch_text = tok.batch_decode(out, skip_special_tokens=True)
        outputs.extend([x.strip() for x in batch_text])
    return outputs


def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": float(bleu), "chrf": float(chrf)}


## Metrics Logger

We keep prompted baselines in a separate CSV to avoid polluting the main training-run log.


In [11]:

PROMPT_METRICS_PATH = os.path.join(PROMPT_RESULTS_DIR, "prompting_metrics.csv")

PROMPT_FIELDS = [
    "timestamp",
    "run_name",
    "model_name",
    "n_shots",
    "split",
    "bleu",
    "chrf",
    "n_eval",
    "gen_kwargs",
    "notes",
]


def log_prompt_metrics(run_name, model_name, n_shots, split, metrics, n_eval, gen_kwargs=None, notes=None):
    row = {
        "timestamp": datetime.now(ZoneInfo("America/Los_Angeles")).isoformat(timespec="seconds"),
        "run_name": run_name,
        "model_name": model_name,
        "n_shots": n_shots,
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "n_eval": n_eval,
        "gen_kwargs": json.dumps(gen_kwargs or {}),
        "notes": notes,
    }
    df_row = pd.DataFrame([row], columns=PROMPT_FIELDS)
    if os.path.exists(PROMPT_METRICS_PATH):
        df_old = pd.read_csv(PROMPT_METRICS_PATH)
        df_new = pd.concat([df_old, df_row], ignore_index=True)
    else:
        df_new = df_row
    df_new.to_csv(PROMPT_METRICS_PATH, index=False)
    return df_row


## Run Zero-Shot and Few-Shot Prompting

This loop:
- builds prompts for each evaluation sentence
- runs generation
- scores BLEU / chrF on the idiom subset
- exports predictions for later manual inspection / annotation


In [12]:

all_run_summaries = []

for n_shots in SHOT_COUNTS:
    prompts = []
    for idx, row in idiom_eval_df.iterrows():
        demos = sample_demos(idiom_train_df, n_shots, exclude_src=row["src"], seed=SEED + idx)
        prompts.append(build_prompt(row["src"], demos=demos))

    preds = generate_texts(prompts, gen_kwargs=GEN_KWARGS)
    refs = idiom_eval_df["tgt"].tolist()
    metrics = compute_metrics(preds, refs)

    run_name = f"{RUN_PREFIX}_{n_shots}shot"
    print(run_name, metrics)

    pred_df = idiom_eval_df.copy()
    pred_df["prompt"] = prompts
    pred_df["pred"] = preds
    pred_df["model_name"] = MODEL_NAME
    pred_df["n_shots"] = n_shots
    pred_path = os.path.join(PROMPT_RESULTS_DIR, f"{run_name}_preds.csv")
    pred_df.to_csv(pred_path, index=False)

    log_prompt_metrics(
        run_name=run_name,
        model_name=MODEL_NAME,
        n_shots=n_shots,
        split="idioms_eval",
        metrics=metrics,
        n_eval=len(pred_df),
        gen_kwargs=GEN_KWARGS,
        notes="few-shot prompted baseline on idiom subset"
    )

    all_run_summaries.append({
        "run_name": run_name,
        "model_name": MODEL_NAME,
        "n_shots": n_shots,
        **metrics,
        "pred_path": pred_path,
    })

summary_df = pd.DataFrame(all_run_summaries)
summary_df


mt5_small_prompt_0shot {'bleu': 0.14554018215556527, 'chrf': 1.9670711421392588}
mt5_small_prompt_3shot {'bleu': 0.14554018215556527, 'chrf': 1.9670711421392588}
mt5_small_prompt_5shot {'bleu': 0.1451688013485524, 'chrf': 2.168259968222038}


,run_name,model_name,n_shots,bleu,chrf,pred_path
0,mt5_small_prompt_0shot,google/mt5-small,0,0.145540,1.967071,/content/drive/MyDrive/ds266_idiom_mt/results/...
1,mt5_small_prompt_3shot,google/mt5-small,3,0.145540,1.967071,/content/drive/MyDrive/ds266_idiom_mt/results/...
2,mt5_small_prompt_5shot,google/mt5-small,5,0.145169,2.168260,/content/drive/MyDrive/ds266_idiom_mt/results/...


## Inspect Sample Outputs

Read through these carefully. Because idioms are semantically tricky, automatic metrics alone are not enough.


In [13]:

SAMPLE_RUN = summary_df.iloc[0]["run_name"]
SAMPLE_PATH = os.path.join(PROMPT_RESULTS_DIR, f"{SAMPLE_RUN}_preds.csv")

sample_df = pd.read_csv(SAMPLE_PATH)
sample_df[["src", "tgt", "pred"]].head(10)


,src,tgt,pred
0,I'm as fit as a fiddle-with energy to spare.,Ich bin fit wie ein Turnschuh - und habe noch ...,<extra_id_0>.
1,Those cheap little metal cars are a dime a dozen.,Diese billigen kleinen Metallautos gibt es wie...,<extra_id_0>.
2,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,<extra_id_0>.
3,How long does it take for the wheel of fashion...,"Wie lange dauert es, bis sich der Kreis in der...",<extra_id_0>.
4,"For a sustained economic recovery, the US cons...",Für einen nachhaltigen Wirtschaftsaufschwung m...,<extra_id_0>.
5,"He fell apart after his wife died, I told him ...","Nach dem Tod seiner Frau brach er zusammen, un...",<extra_id_0>.
6,"He really shot himself in the foot, telling th...","Er hat sich selbst ins Knie geschossen, als er...",<extra_id_0>.
7,"I don't want to put the cart before the horse,...",Ich möchte das Pferd nicht von hinten aufzäume...,<extra_id_0>.
8,New bank competitors have struggled to get off...,"Neue Bankkonkurrenten hatten es schwer, auf di...",<extra_id_0>.
9,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,<extra_id_0>.


## Exploratory Diagnostics for mT5

In [14]:
def load_pred_csv(run_name, results_dir=PROMPT_RESULTS_DIR):
    path = os.path.join(results_dir, f"{run_name}_preds.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing prediction file: {path}")
    return pd.read_csv(path)


def preview_predictions(pred_df, n=10):
    cols = [c for c in ["src", "tgt", "pred"] if c in pred_df.columns]
    display(pred_df[cols].head(n))


def diagnose_outputs(preds, label="run"):
    preds = ["" if pd.isna(p) else str(p) for p in preds]
    stripped = [p.strip() for p in preds]

    lengths = [len(p) for p in stripped]
    unique_ratio = len(set(stripped)) / max(len(stripped), 1)
    empty_count = sum(1 for p in stripped if len(p) == 0)

    print(f"=== Diagnostics: {label} ===")
    print("n_outputs:", len(stripped))
    print("avg_len:", round(float(np.mean(lengths)), 2) if lengths else 0)
    print("min_len:", int(np.min(lengths)) if lengths else 0)
    print("max_len:", int(np.max(lengths)) if lengths else 0)
    print("empty_count:", empty_count)
    print("unique_ratio:", round(unique_ratio, 4))
    print()

In [15]:
RUN_0 = f"{RUN_PREFIX}_0shot"
RUN_3 = f"{RUN_PREFIX}_3shot"
RUN_5 = f"{RUN_PREFIX}_5shot"

df0 = load_pred_csv(RUN_0)
df3 = load_pred_csv(RUN_3)
df5 = load_pred_csv(RUN_5)

diagnose_outputs(df0["pred"].tolist(), label=RUN_0)
diagnose_outputs(df3["pred"].tolist(), label=RUN_3)
diagnose_outputs(df5["pred"].tolist(), label=RUN_5)

=== Diagnostics: mt5_small_prompt_0shot ===
n_outputs: 25
avg_len: 13.0
min_len: 13
max_len: 13
empty_count: 0
unique_ratio: 0.04

=== Diagnostics: mt5_small_prompt_3shot ===
n_outputs: 25
avg_len: 13.0
min_len: 13
max_len: 13
empty_count: 0
unique_ratio: 0.04

=== Diagnostics: mt5_small_prompt_5shot ===
n_outputs: 25
avg_len: 13.84
min_len: 13
max_len: 24
empty_count: 0
unique_ratio: 0.16



In [16]:
def compare_prompt_runs(df0, df3, df5, n=10):
    out = pd.DataFrame({
        "src": df0["src"].head(n),
        "ref": df0["tgt"].head(n),
        "pred_0shot": df0["pred"].head(n),
        "pred_3shot": df3["pred"].head(n),
        "pred_5shot": df5["pred"].head(n),
    })
    return out

compare_df = compare_prompt_runs(df0, df3, df5, n=10)
display(compare_df)

,src,ref,pred_0shot,pred_3shot,pred_5shot
0,I'm as fit as a fiddle-with energy to spare.,Ich bin fit wie ein Turnschuh - und habe noch ...,<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
1,Those cheap little metal cars are a dime a dozen.,Diese billigen kleinen Metallautos gibt es wie...,<extra_id_0>.,<extra_id_0>.,<extra_id_0> ersticken.
2,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
3,How long does it take for the wheel of fashion...,"Wie lange dauert es, bis sich der Kreis in der...",<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
4,"For a sustained economic recovery, the US cons...",Für einen nachhaltigen Wirtschaftsaufschwung m...,<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
5,"He fell apart after his wife died, I told him ...","Nach dem Tod seiner Frau brach er zusammen, un...",<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
6,"He really shot himself in the foot, telling th...","Er hat sich selbst ins Knie geschossen, als er...",<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
7,"I don't want to put the cart before the horse,...",Ich möchte das Pferd nicht von hinten aufzäume...,<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
8,New bank competitors have struggled to get off...,"Neue Bankkonkurrenten hatten es schwer, auf di...",<extra_id_0>.,<extra_id_0>.,<extra_id_0>.
9,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,<extra_id_0>.,<extra_id_0>.,<extra_id_0>.


In [17]:
def show_most_common_predictions(pred_df, top_k=10):
    vc = pred_df["pred"].fillna("").astype(str).str.strip().value_counts().head(top_k)
    display(vc)

print("Most common outputs for 0-shot")
show_most_common_predictions(df0, top_k=10)

print("Most common outputs for 3-shot")
show_most_common_predictions(df3, top_k=10)

print("Most common outputs for 5-shot")
show_most_common_predictions(df5, top_k=10)

Most common outputs for 0-shot


,count
pred,
<extra_id_0>.,25


Most common outputs for 3-shot


,count
pred,
<extra_id_0>.,25


Most common outputs for 5-shot


,count
pred,
<extra_id_0>.,22
<extra_id_0> ersticken.,1
<extra_id_0>:,1
<extra_id_0> verbessern.,1


### Third T5 prompt builder to diagnose the "extra_id_0" issue

In [18]:
def build_prompt_t5_style(src_text, demos=None):
    demos = demos or []

    blocks = []
    for ex in demos:
        blocks.append(
            f"translate English to German: {ex['src']}\n{ex['tgt']}"
        )

    blocks.append(
        f"translate English to German: {src_text}"
    )

    return "\n\n".join(blocks)

In [19]:
# === Quick T5-style debug run (mT5) ===

DEBUG_N = 10

if DEBUG_N is not None:
    debug_eval_df = idiom_eval_df.head(DEBUG_N).copy()
else:
    debug_eval_df = idiom_eval_df.copy()

print("Running T5-style debug on:", len(debug_eval_df), "examples")


def run_t5_style_debug(n_shots):
    prompts = []

    for idx, row in debug_eval_df.iterrows():
        demos = sample_demos(
            idiom_train_df,
            n_shots,
            exclude_src=row["src"],
            seed=SEED + idx
        )
        prompt = build_prompt_t5_style(row["src"], demos=demos)
        prompts.append(prompt)

    # greedy decoding
    gen_kwargs = {
        "max_new_tokens": 64,
        "do_sample": False,
    }

    preds = generate_texts(prompts, gen_kwargs=gen_kwargs)
    refs = debug_eval_df["tgt"].tolist()
    metrics = compute_metrics(preds, refs)

    print(f"\n=== T5-style | {n_shots}-shot ===")
    print("metrics:", metrics)
    diagnose_outputs(preds, label=f"T5-style-{n_shots}shot")

    preview_df = pd.DataFrame({
        "src": debug_eval_df["src"],
        "ref": refs,
        "pred": preds,
    })

    display(preview_df.head(10))

    return preds


# Run 0-shot and 3-shot
preds_0 = run_t5_style_debug(0)
preds_3 = run_t5_style_debug(3)

Running T5-style debug on: 10 examples

=== T5-style | 0-shot ===
metrics: {'bleu': 0.0, 'chrf': 1.6121495781536717}
=== Diagnostics: T5-style-0shot ===
n_outputs: 10
avg_len: 12.1
min_len: 12
max_len: 13
empty_count: 0
unique_ratio: 0.2



,src,ref,pred
0,I'm as fit as a fiddle-with energy to spare.,Ich bin fit wie ein Turnschuh - und habe noch ...,<extra_id_0>
1,Those cheap little metal cars are a dime a dozen.,Diese billigen kleinen Metallautos gibt es wie...,<extra_id_0>
2,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,<extra_id_0>.
3,How long does it take for the wheel of fashion...,"Wie lange dauert es, bis sich der Kreis in der...",<extra_id_0>
4,"For a sustained economic recovery, the US cons...",Für einen nachhaltigen Wirtschaftsaufschwung m...,<extra_id_0>
5,"He fell apart after his wife died, I told him ...","Nach dem Tod seiner Frau brach er zusammen, un...",<extra_id_0>
6,"He really shot himself in the foot, telling th...","Er hat sich selbst ins Knie geschossen, als er...",<extra_id_0>
7,"I don't want to put the cart before the horse,...",Ich möchte das Pferd nicht von hinten aufzäume...,<extra_id_0>
8,New bank competitors have struggled to get off...,"Neue Bankkonkurrenten hatten es schwer, auf di...",<extra_id_0>
9,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,<extra_id_0>



=== T5-style | 3-shot ===
metrics: {'bleu': 0.2504818895710715, 'chrf': 2.2122167568620617}
=== Diagnostics: T5-style-3shot ===
n_outputs: 10
avg_len: 15.1
min_len: 13
max_len: 26
empty_count: 0
unique_ratio: 0.3



,src,ref,pred
0,I'm as fit as a fiddle-with energy to spare.,Ich bin fit wie ein Turnschuh - und habe noch ...,<extra_id_0>.
1,Those cheap little metal cars are a dime a dozen.,Diese billigen kleinen Metallautos gibt es wie...,<extra_id_0> zu ersticken.
2,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,<extra_id_0>.
3,How long does it take for the wheel of fashion...,"Wie lange dauert es, bis sich der Kreis in der...",<extra_id_0>.
4,"For a sustained economic recovery, the US cons...",Für einen nachhaltigen Wirtschaftsaufschwung m...,<extra_id_0>.
5,"He fell apart after his wife died, I told him ...","Nach dem Tod seiner Frau brach er zusammen, un...",<extra_id_0> erfahren
6,"He really shot himself in the foot, telling th...","Er hat sich selbst ins Knie geschossen, als er...",<extra_id_0>.
7,"I don't want to put the cart before the horse,...",Ich möchte das Pferd nicht von hinten aufzäume...,<extra_id_0>.
8,New bank competitors have struggled to get off...,"Neue Bankkonkurrenten hatten es schwer, auf di...",<extra_id_0>.
9,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,<extra_id_0>.


### Interpretation of mT5 exploratory results

mT5-small consistently degenerated to sentinel-token outputs such as `<extra_id_0>` across multiple prompt styles, including instruction-style, pattern-only, and T5-style task-prefix prompting. This suggests that multilingual pretraining alone is insufficient for our few-shot idiom translation setup, and motivates moving to an instruction-tuned model family.

## Save Summary Artifact


In [20]:

summary_path = os.path.join(PROMPT_RESULTS_DIR, f"{RUN_PREFIX}_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("saved:", summary_path)
print("metrics log:", PROMPT_METRICS_PATH)


saved: /content/drive/MyDrive/ds266_idiom_mt/results/prompting/mt5_small_prompt_summary.csv
metrics log: /content/drive/MyDrive/ds266_idiom_mt/results/prompting/prompting_metrics.csv
